<a href="https://colab.research.google.com/github/mrdbourke/pytorch-deep-learning/blob/main/extras/solutions/05_pytorch_going_modular_exercise_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. PyTorch 模块化实战练习解答

欢迎来到 05 章 PyTorch 模块化实战练习解答 notebook。

> **注意：** 每道练习通常不止一种解法，本 notebook 仅展示其中一种可行方案。

## 资源

1. 这些练习/解答基于 Zero to Mastery 的 Learn PyTorch for Deep Learning 课程中 [05. PyTorch Going Modular](https://www.learnpytorch.io/05_pytorch_going_modular/) 章节。
2. 可观看 YouTube 上的[完整解题实战（含报错与调试过程）](https://youtu.be/ijgFhMK3pp4)。
3. 更多解答见课程 GitHub 的 [solutions 目录](https://github.com/mrdbourke/pytorch-deep-learning/tree/main/extras/solutions)。

## 1. 将“获取数据”（第 1 节 Get Data）的代码改写为 Python 脚本，例如 `get_data.py`。

* 运行 `python get_data.py` 时，脚本应先检查数据是否已存在；若存在则跳过下载。
* 若下载成功，你应能从 `data` 目录访问 `pizza_steak_sushi` 图像数据。

In [ ]:
%%writefile get_data.py
import os
import zipfile

from pathlib import Path

import requests

# 设置数据目录路径
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

# 如果图像目录不存在，则下载并准备数据
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)
    
# 下载 pizza、steak、sushi 数据
with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
    print("Downloading pizza, steak, sushi data...")
    f.write(request.content)

# 解压 pizza、steak、sushi 数据
with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
    print("Unzipping pizza, steak, sushi data...") 
    zip_ref.extractall(image_path)

# 删除压缩包
os.remove(data_path / "pizza_steak_sushi.zip")

Writing get_data.py


In [ ]:
!python get_data.py

Did not find data/pizza_steak_sushi directory, creating one...
Unzipping pizza, steak, sushi data...


## 2. 使用 [Python 的 `argparse` 模块](https://docs.python.org/3/library/argparse.html) 为 `train.py` 传入自定义训练超参数。
* 为下列项添加参数开关：
  * 训练/测试目录
  * 学习率
  * batch size
  * 训练 epoch 数
  * TinyVGG 模型中的隐藏单元数
    * 以上参数的默认值保持与 notebook 05 一致。
* 例如，你应能运行类似如下命令：`python train.py --learning_rate 0.003 batch_size 64 num_epochs 20`，以学习率 0.003、batch size 64 训练 20 个 epoch。
* **注意：** `train.py` 依赖你在 05 章创建的其他脚本（如 `model_builder.py`、`utils.py`、`engine.py`），请确保它们可用。相关文件可在课程 GitHub 的 [`going_modular` 文件夹](https://github.com/mrdbourke/pytorch-deep-learning/tree/main/going_modular/going_modular)中找到。

In [ ]:
%%writefile data_setup.py
"""
包含用于创建图像分类数据 DataLoader 的功能。
"""
import os

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()

def create_dataloaders(
    train_dir: str, 
    test_dir: str, 
    transform: transforms.Compose, 
    batch_size: int, 
    num_workers: int=NUM_WORKERS
):
  """创建训练与测试 DataLoader。
  接收训练目录与测试目录路径，先转换为 PyTorch Dataset，
  再转换为 PyTorch DataLoader。
  参数:
    train_dir: 训练数据目录路径。
    test_dir: 测试数据目录路径。
    transform: 作用在训练与测试数据上的 torchvision transform。
    batch_size: 每个 DataLoader 中每个 batch 的样本数。
    num_workers: 每个 DataLoader 使用的 worker 数。
  返回:
    一个元组 (train_dataloader, test_dataloader, class_names)。
    其中 class_names 是目标类别名称列表。
    示例用法:
      train_dataloader, test_dataloader, class_names = \
        = create_dataloaders(train_dir=path/to/train_dir,
                             test_dir=path/to/test_dir,
                             transform=some_transform,
                             batch_size=32,
                             num_workers=4)
  """
  # 使用 ImageFolder 创建数据集
  train_data = datasets.ImageFolder(train_dir, transform=transform)
  test_data = datasets.ImageFolder(test_dir, transform=transform)

  # 获取类别名称
  class_names = train_data.classes

  # 将图像转换为 DataLoader
  train_dataloader = DataLoader(
      train_data,
      batch_size=batch_size,
      shuffle=True,
      num_workers=num_workers,
      pin_memory=True,
  )
  test_dataloader = DataLoader(
      test_data,
      batch_size=batch_size,
      shuffle=False,
      num_workers=num_workers,
      pin_memory=True,
  )

  return train_dataloader, test_dataloader, class_names

Writing data_setup.py


In [ ]:
%%writefile engine.py
"""
包含训练和测试 PyTorch 模型的函数。
"""
import torch

from tqdm.auto import tqdm
from typing import Dict, List, Tuple

def train_step(model: torch.nn.Module, 
               dataloader: torch.utils.data.DataLoader, 
               loss_fn: torch.nn.Module, 
               optimizer: torch.optim.Optimizer,
               device: torch.device) -> Tuple[float, float]:
    """训练 PyTorch 模型一个 epoch。
    将目标模型切换到训练模式，然后执行完整训练流程
    （前向传播、损失计算、优化器更新）。
    参数:
    model: 待训练的 PyTorch 模型。
    dataloader: 用于训练的数据 DataLoader。
    loss_fn: 需要最小化的损失函数。
    optimizer: 用于最小化损失函数的优化器。
    device: 计算设备（如 "cuda" 或 "cpu"）。
    返回:
    一个包含训练损失和训练准确率的元组。
    形式为 (train_loss, train_accuracy)，例如:
    (0.1112, 0.8743)
    """
    # 将模型切换到训练模式
    model.train()

    # 初始化训练损失和训练准确率
    train_loss, train_acc = 0, 0

    # 遍历 data loader 中的每个 batch
    for batch, (X, y) in enumerate(dataloader):
        # 将数据发送到目标设备
        X, y = X.to(device), y.to(device)

        # 1. 前向传播
        y_pred = model(X)

        # 2. 计算并累积损失
        loss = loss_fn(y_pred, y)
        train_loss += loss.item() 

        # 3. 梯度清零
        optimizer.zero_grad()

        # 4. 反向传播
        loss.backward()

        # 5. 参数更新
        optimizer.step()

        # 计算并累积所有 batch 的准确率指标
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)

    # 计算每个 batch 的平均损失与准确率
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc

def test_step(model: torch.nn.Module, 
              dataloader: torch.utils.data.DataLoader, 
              loss_fn: torch.nn.Module,
              device: torch.device) -> Tuple[float, float]:
    """测试 PyTorch 模型一个 epoch。
    将目标模型切换到评估模式（"eval"），然后在测试集上做前向传播。
    参数:
    model: 待测试的 PyTorch 模型。
    dataloader: 用于测试的数据 DataLoader。
    loss_fn: 在测试数据上计算损失的损失函数。
    device: 计算设备（如 "cuda" 或 "cpu"）。
    返回:
    一个包含测试损失和测试准确率的元组。
    形式为 (test_loss, test_accuracy)，例如:
    (0.0223, 0.8985)
    """
    # 将模型切换到评估模式
    model.eval() 

    # 初始化测试损失和测试准确率
    test_loss, test_acc = 0, 0

    # 开启推理上下文管理器
    with torch.inference_mode():
        # 遍历 DataLoader 中的每个 batch
        for batch, (X, y) in enumerate(dataloader):
            # 将数据发送到目标设备
            X, y = X.to(device), y.to(device)

            # 1. 前向传播
            test_pred_logits = model(X)

            # 2. 计算并累积损失
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            # 计算并累积准确率
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))

    # 计算每个 batch 的平均损失与准确率
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc

def train(model: torch.nn.Module, 
          train_dataloader: torch.utils.data.DataLoader, 
          test_dataloader: torch.utils.data.DataLoader, 
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device) -> Dict[str, List]:
    """训练并测试一个 PyTorch 模型。
    在每个 epoch 中依次调用 train_step() 和 test_step()，
    完成同一轮中的训练与测试，并记录与打印评估指标。
    参数:
    model: 待训练与测试的 PyTorch 模型。
    train_dataloader: 训练数据 DataLoader。
    test_dataloader: 测试数据 DataLoader。
    optimizer: 用于最小化损失函数的优化器。
    loss_fn: 在训练/测试中用于计算损失的损失函数。
    epochs: 训练轮数。
    device: 计算设备（如 "cuda" 或 "cpu"）。
    返回:
    一个包含训练/测试损失和训练/测试准确率的字典，
    每个指标都按 epoch 存在一个列表中。
    形式: {train_loss: [...],
              train_acc: [...],
              test_loss: [...],
              test_acc: [...]} 
    例如 epochs=2 时:
             {train_loss: [2.0616, 1.0537],
              train_acc: [0.3945, 0.3945],
              test_loss: [1.2641, 1.5706],
              test_acc: [0.3400, 0.2973]} 
    """
    # 创建空结果字典
    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []
    }

    # 按 epoch 循环执行训练与测试步骤
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                          dataloader=train_dataloader,
                                          loss_fn=loss_fn,
                                          optimizer=optimizer,
                                          device=device)
        test_loss, test_acc = test_step(model=model,
          dataloader=test_dataloader,
          loss_fn=loss_fn,
          device=device)

        # 打印训练进度
        print(
          f"Epoch: {epoch+1} | "
          f"train_loss: {train_loss:.4f} | "
          f"train_acc: {train_acc:.4f} | "
          f"test_loss: {test_loss:.4f} | "
          f"test_acc: {test_acc:.4f}"
        )

        # 更新结果字典
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

    # 在所有 epoch 结束后返回结果
    return results

Writing engine.py


In [ ]:
%%writefile model_builder.py
"""
包含用于实例化 TinyVGG 模型的 PyTorch 代码。
"""
import torch
from torch import nn 

class TinyVGG(nn.Module):
    """创建 TinyVGG 架构。
    在 PyTorch 中复现 CNN explainer 网站中的 TinyVGG 架构。
    原始架构见: https://poloclub.github.io/cnn-explainer/
    参数:
    input_shape: 输入通道数。
    hidden_units: 层间隐藏单元数量。
    output_shape: 输出单元数量。
    """
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
        super().__init__()
        self.conv_block_1 = nn.Sequential(
          nn.Conv2d(in_channels=input_shape, 
                    out_channels=hidden_units, 
                    kernel_size=3, 
                    stride=1, 
                    padding=0),  
          nn.ReLU(),
          nn.Conv2d(in_channels=hidden_units, 
                    out_channels=hidden_units,
                    kernel_size=3,
                    stride=1,
                    padding=0),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2,
                        stride=2)
        )
        self.conv_block_2 = nn.Sequential(
          nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
          nn.ReLU(),
          nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
          nn.ReLU(),
          nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
          nn.Flatten(),
          # 这里的 in_features 形状是怎么来的？
          # 因为网络每一层都会压缩并改变输入数据的形状。
          nn.Linear(in_features=hidden_units*13*13,
                    out_features=output_shape)
        )
    
    def forward(self, x: torch.Tensor):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.classifier(x)
        return x
        # return self.classifier(self.block_2(self.block_1(x))) # <- 利用算子融合的优势

Writing model_builder.py


In [ ]:
%%writefile utils.py
"""
包含用于 PyTorch 模型训练与保存的各类工具函数。
"""
import torch
from pathlib import Path

def save_model(model: torch.nn.Module,
               target_dir: str,
               model_name: str):
    """将 PyTorch 模型保存到目标目录。
    参数:
    model: 要保存的 PyTorch 模型。
    target_dir: 保存模型的目录。
    model_name: 保存后的模型文件名。
      扩展名应为 ".pth" 或 ".pt"。
    示例:
    save_model(model=model_0,
               target_dir="models",
               model_name="05_going_modular_tingvgg_model.pth")
    """
    # 创建目标目录
    target_dir_path = Path(target_dir)
    target_dir_path.mkdir(parents=True,
                        exist_ok=True)

    # 创建模型保存路径
    assert model_name.endswith(".pth") or model_name.endswith(".pt"), "model_name should end with '.pt' or '.pth'"
    model_save_path = target_dir_path / model_name

    # 保存模型 state_dict()
    print(f"[INFO] Saving model to: {model_save_path}")
    torch.save(obj=model.state_dict(),
             f=model_save_path)

Writing utils.py


In [ ]:
%%writefile train.py
"""
使用设备无关代码训练 PyTorch 图像分类模型。
"""

import os
import argparse

import torch

from torchvision import transforms

import data_setup, engine, model_builder, utils

# 创建参数解析器
parser = argparse.ArgumentParser(description="Get some hyperparameters.")

# 添加 num_epochs 参数
parser.add_argument("--num_epochs", 
                     default=10, 
                     type=int, 
                     help="the number of epochs to train for")

# 添加 batch_size 参数
parser.add_argument("--batch_size",
                    default=32,
                    type=int,
                    help="number of samples per batch")

# 添加 hidden_units 参数
parser.add_argument("--hidden_units",
                    default=10,
                    type=int,
                    help="number of hidden units in hidden layers")

# 添加 learning_rate 参数
parser.add_argument("--learning_rate",
                    default=0.001,
                    type=float,
                    help="learning rate to use for model")

# 添加训练目录参数
parser.add_argument("--train_dir",
                    default="data/pizza_steak_sushi/train",
                    type=str,
                    help="directory file path to training data in standard image classification format")

# 添加测试目录参数
parser.add_argument("--test_dir",
                    default="data/pizza_steak_sushi/test",
                    type=str,
                    help="directory file path to testing data in standard image classification format")

# 从解析器中读取参数
args = parser.parse_args()

# 设置超参数
NUM_EPOCHS = args.num_epochs
BATCH_SIZE = args.batch_size
HIDDEN_UNITS = args.hidden_units
LEARNING_RATE = args.learning_rate
print(f"[INFO] Training a model for {NUM_EPOCHS} epochs with batch size {BATCH_SIZE} using {HIDDEN_UNITS} hidden units and a learning rate of {LEARNING_RATE}")

# 设置目录
train_dir = args.train_dir
test_dir = args.test_dir
print(f"[INFO] Training data file: {train_dir}")
print(f"[INFO] Testing data file: {test_dir}")

# 设置目标设备
device = "cuda" if torch.cuda.is_available() else "cpu"

# 创建 transforms
data_transform = transforms.Compose([
  transforms.Resize((64, 64)),
  transforms.ToTensor()
])

# 借助 data_setup.py 创建 DataLoader
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=data_transform,
    batch_size=BATCH_SIZE
)

# 借助 model_builder.py 创建模型
model = model_builder.TinyVGG(
    input_shape=3,
    hidden_units=HIDDEN_UNITS,
    output_shape=len(class_names)
).to(device)

# 设置损失函数和优化器
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),
                             lr=LEARNING_RATE)

# 借助 engine.py 开始训练
engine.train(model=model,
             train_dataloader=train_dataloader,
             test_dataloader=test_dataloader,
             loss_fn=loss_fn,
             optimizer=optimizer,
             epochs=NUM_EPOCHS,
             device=device)

# 借助 utils.py 保存模型
utils.save_model(model=model,
                 target_dir="models",
                 model_name="05_going_modular_script_mode_tinyvgg_model.pth")

Writing train.py


In [ ]:
!python train.py --num_epochs 5 --batch_size 128 --hidden_units 128 --learning_rate 0.0003

[INFO] Training a model for 5 epochs with batch size 128 using 128 hidden units and a learning rate of 0.0003
[INFO] Training data file: data/pizza_steak_sushi/train
[INFO] Testing data file: data/pizza_steak_sushi/test
  0% 0/5 [00:00<?, ?it/s]Epoch: 1 | train_loss: 1.0994 | train_acc: 0.3082 | test_loss: 1.0933 | test_acc: 0.3333
 20% 1/5 [00:01<00:07,  1.94s/it]Epoch: 2 | train_loss: 1.0897 | train_acc: 0.3859 | test_loss: 1.0803 | test_acc: 0.3733
 40% 2/5 [00:03<00:04,  1.65s/it]Epoch: 3 | train_loss: 1.0683 | train_acc: 0.3926 | test_loss: 1.0466 | test_acc: 0.4800
 60% 3/5 [00:04<00:03,  1.56s/it]Epoch: 4 | train_loss: 1.0318 | train_acc: 0.4898 | test_loss: 1.0320 | test_acc: 0.4267
 80% 4/5 [00:06<00:01,  1.55s/it]Epoch: 5 | train_loss: 0.9779 | train_acc: 0.5560 | test_loss: 1.0151 | test_acc: 0.4000
100% 5/5 [00:07<00:00,  1.57s/it]
[INFO] Saving model to: models/05_going_modular_script_mode_tinyvgg_model.pth


## 3. 编写一个预测脚本（例如 `predict.py`），给定目标图像路径与已保存模型后进行预测。

* 例如，你应能运行 `python predict.py some_image.jpeg`，并让训练好的 PyTorch 模型输出该图像的预测结果。
* 预测代码可参考 [notebook 04 中“对自定义图像进行预测”章节](https://www.learnpytorch.io/04_pytorch_custom_datasets/#113-putting-custom-image-prediction-together-building-a-function)。
* 你可能还需要编写加载已训练模型的代码。

In [ ]:
%%writefile predict.py
import torch
import torchvision
import argparse

import model_builder

# 创建参数解析器
parser = argparse.ArgumentParser()

# 获取图像路径参数
parser.add_argument("--image",
                    help="target image filepath to predict on")

# 获取模型路径参数
parser.add_argument("--model_path",
                    default="models/05_going_modular_script_mode_tinyvgg_model.pth",
                    type=str,
                    help="target model to use for prediction filepath")

args = parser.parse_args()

# 设置类别名称
class_names = ["pizza", "steak", "sushi"]

# 设置设备
device = "cuda" if torch.cuda.is_available() else "cpu"

# 获取图像路径
IMG_PATH = args.image
print(f"[INFO] Predicting on {IMG_PATH}")

# 加载模型的函数
def load_model(filepath=args.model_path):
  # 必须与保存模型时使用相同的超参数
  model = model_builder.TinyVGG(input_shape=3,
                                hidden_units=128,
                                output_shape=3).to(device)

  print(f"[INFO] Loading in model from: {filepath}")
  # 从文件加载保存的模型 state_dict
  model.load_state_dict(torch.load(filepath))

  return model

# 加载模型并对指定图像进行预测
def predict_on_image(image_path=IMG_PATH, filepath=args.model_path):
  # 加载模型
  model = load_model(filepath)

  # 读取图像并转换为 torch.float32（与模型类型一致）
  image = torchvision.io.read_image(str(IMG_PATH)).type(torch.float32)

  # 预处理图像到 0-1 范围
  image = image / 255.

  # 将图像缩放到与模型一致的尺寸
  transform = torchvision.transforms.Resize(size=(64, 64))
  image = transform(image) 

  # 对图像进行预测
  model.eval()
  with torch.inference_mode():
    # 将图像发送到目标设备
    image = image.to(device)

    # 获取预测 logits
    pred_logits = model(image.unsqueeze(dim=0)) # 确保图像包含 batch 维度

    # 获取预测概率
    pred_prob = torch.softmax(pred_logits, dim=1)

    # 获取预测标签
    pred_label = torch.argmax(pred_prob, dim=1)
    pred_label_class = class_names[pred_label]

  print(f"[INFO] Pred class: {pred_label_class}, Pred prob: {pred_prob.max():.3f}")

if __name__ == "__main__":
  predict_on_image()

Writing predict.py


In [ ]:
!python predict.py --image data/pizza_steak_sushi/test/sushi/175783.jpg

[INFO] Predicting on data/pizza_steak_sushi/test/sushi/175783.jpg
[INFO] Loading in model from: models/05_going_modular_script_mode_tinyvgg_model.pth
[INFO] Pred class: steak, Pred prob: 0.376
